# DNAffinity EDA handoff: validated SELEX descriptor tables

This notebook prepares the selected HT-SELEX 10-mer datasets for later machine-learning experiments. It loads the original TF cycle tables and canonical clean tetramer/hexamer descriptors, validates their identifiers and values, maps overlapping windows, averages descriptor values per 10-mer, and exports one compressed table per TF.

The notebook follows the existing EDA methodology. It does not scale features, filter observations, fit models, or change source data. Affinity, ObservedCount and SE are preserved as read from the source; Affinity remains the target for later work.

## 1. Reproducibility and inputs

Run the notebook from any directory inside the repository; the project root is located from the notebook path, not from a personal absolute path. Required input files and their purposes are listed in `notebooks/README_handoff.md`. If a required file is absent, the notebook stops and reports the missing path rather than substituting another table or cycle.

The selected cycle for each TF follows the completed Shape–Affinity EDA. TF files are read one at a time and in chunks to limit memory use.

In [3]:
from pathlib import Path
from itertools import product
import gzip
import gc
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "src" / "feature_config.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the DNAffinity repository root (src/feature_config.py).")
sys.path.insert(0, str(ROOT))

from src.feature_config import (
    FEATURE_GROUPS_CLEAN_TETRAMER,
    FEATURE_GROUPS_CLEAN_HEXAMER,
    FEATURES_CLEAN_TETRAMER,
    FEATURES_CLEAN_HEXAMER,
)

DATA_DIR = ROOT / "data" / "selex"
TF_DIR = DATA_DIR / "transcription_factors"
OUTPUT_DIR = ROOT / "outputs" / "selex_ml_ready"
CHUNK_SIZE = 50_000


### 1.1. Selected HT-SELEX source files

These are the same TF-cycle files used in the completed analysis. `Kmer` is the 10-mer sequence identifier. Some files contain an extra leading index column; the workflow selects required columns by name, so that column is not treated as a feature.

In [5]:
TF_CYCLES = {
    "Gata4": 4,
    "Zfp24": 5,
    "arid3": 6,
    "hoxd1": 3,
    "nkx3": 6,
    "pax3-like": 5,
    "six1": 5,
    "tbx15": 5,
    "telf": 5,
    "uncxb": 5,
    "xbpc": 5,
}
TARGET_COLUMNS = ["Kmer", "ObservedCount", "Affinity", "SE"]
source_paths = {
    tf: TF_DIR / tf / f"{cycle}.txt"
    for tf, cycle in TF_CYCLES.items()
}
missing_sources = [str(path.relative_to(ROOT)) for path in source_paths.values() if not path.is_file()]
if missing_sources:
    raise FileNotFoundError("Missing selected SELEX input(s):\n" + "\n".join(missing_sources))

display(pd.DataFrame([
    {"TF": tf, "Cycle": cycle, "Input file": str(source_paths[tf].relative_to(ROOT))}
    for tf, cycle in TF_CYCLES.items()
]))


,TF,Cycle,Input file
0,Gata4,4,data/selex/transcription_factors/Gata4/4.txt
1,Zfp24,5,data/selex/transcription_factors/Zfp24/5.txt
2,arid3,6,data/selex/transcription_factors/arid3/6.txt
3,hoxd1,3,data/selex/transcription_factors/hoxd1/3.txt
4,nkx3,6,data/selex/transcription_factors/nkx3/6.txt
5,pax3-like,5,data/selex/transcription_factors/pax3-like/5.txt
6,six1,5,data/selex/transcription_factors/six1/5.txt
7,tbx15,5,data/selex/transcription_factors/tbx15/5.txt
8,telf,5,data/selex/transcription_factors/telf/5.txt
9,uncxb,5,data/selex/transcription_factors/uncxb/5.txt


### 1.2. Canonical clean descriptor tables

The clean tables are the source of descriptor values. `feature_config.py` is the single source of truth for feature names, group membership and ordering: 36 tetramer features and 40 hexamer features. This includes 21 independent upper-triangular FCE elements and only the non-redundant electrostatic features. The raw descriptor files are not needed to run this handoff.

In [7]:
tetramer_path = DATA_DIR / "tetramer" / "input_tetramer_clean.csv"
hexamer_path = DATA_DIR / "hexamer" / "input_hexamer_clean.csv"
for path in (tetramer_path, hexamer_path):
    if not path.is_file():
        raise FileNotFoundError(f"Missing required descriptor table: {path.relative_to(ROOT)}")

tetramer_table = pd.read_csv(tetramer_path)
hexamer_table = pd.read_csv(hexamer_path)
display(pd.DataFrame([
    {"Table": "Tetramer", "Rows": len(tetramer_table), "Columns including SEQUENCE": tetramer_table.shape[1], "Canonical features": len(FEATURES_CLEAN_TETRAMER)},
    {"Table": "Hexamer", "Rows": len(hexamer_table), "Columns including SEQUENCE": hexamer_table.shape[1], "Canonical features": len(FEATURES_CLEAN_HEXAMER)},
]))


,Table,Rows,Columns including SEQUENCE,Canonical features
0,Tetramer,256,37,36
1,Hexamer,4096,41,40


## 2. Validate input tables

Validation checks the full expected 4-mer and 6-mer sequence spaces, sequence length and alphabet, duplicate identifiers, missing values, numeric descriptor types, and exact feature-column order. Mapping later uses sequence identifiers encoded as lookup keys; it does not rely on matching row positions.

In [9]:
def validate_descriptor_table(table, k, expected_features, table_name):
    expected_sequences = {"".join(chars) for chars in product("ACGT", repeat=k)}
    sequence_ids = table["SEQUENCE"]
    numeric = table[expected_features]
    checks = {
        "Expected rows": len(table) == 4**k,
        "Unique sequence IDs": not sequence_ids.duplicated().any(),
        "Valid sequence length/alphabet": sequence_ids.str.fullmatch(f"[ACGT]{{{k}}}").all(),
        "Complete sequence coverage": set(sequence_ids) == expected_sequences,
        "Feature names/order match canonical config": table.columns.tolist() == ["SEQUENCE", *expected_features],
        "No missing values": not table[["SEQUENCE", *expected_features]].isna().any().any(),
        "All descriptor columns numeric": all(pd.api.types.is_numeric_dtype(numeric[c]) for c in expected_features),
        "All descriptor values finite": np.isfinite(numeric.to_numpy(dtype=np.float64)).all(),
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise ValueError(f"{table_name} validation failed: {', '.join(failed)}")
    return {"Table": table_name, "Rows": len(table), "Features": len(expected_features), **checks}

descriptor_validation = pd.DataFrame([
    validate_descriptor_table(tetramer_table, 4, FEATURES_CLEAN_TETRAMER, "tetramer"),
    validate_descriptor_table(hexamer_table, 6, FEATURES_CLEAN_HEXAMER, "hexamer"),
])
display(descriptor_validation)


,Table,Rows,Features,Expected rows,Unique sequence IDs,Valid sequence length/alphabet,Complete sequence coverage,Feature names/order match canonical config,No missing values,All descriptor columns numeric,All descriptor values finite
0,tetramer,256,36,True,True,True,True,True,True,True,True
1,hexamer,4096,40,True,True,True,True,True,True,True,True


In [10]:
feature_groups = pd.concat([
    pd.DataFrame([{"Sequence length": "Tetramer", "Descriptor group": group, "Feature": feature}
                  for group, features in FEATURE_GROUPS_CLEAN_TETRAMER.items() for feature in features]),
    pd.DataFrame([{"Sequence length": "Hexamer", "Descriptor group": group, "Feature": feature}
                  for group, features in FEATURE_GROUPS_CLEAN_HEXAMER.items() for feature in features]),
], ignore_index=True)
display(feature_groups.groupby(["Sequence length", "Descriptor group"], as_index=False).agg(Features=("Feature", "count")))


,Sequence length,Descriptor group,Features
0,Hexamer,AVG,6
1,Hexamer,Electrostatic,3
2,Hexamer,FCE,21
3,Hexamer,Ion-related,10
4,Tetramer,AVG,6
5,Tetramer,Electrostatic,3
6,Tetramer,FCE,21
7,Tetramer,Ion-related,6


## 3. Build sequence-keyed descriptor lookups

The EDA maps each DNA word to an integer using a two-bit code per nucleotide (A, C, G, T). We use the same encoding to build lookup arrays, placing descriptor values at the code for their `SEQUENCE`. The completeness checks above ensure every possible window has one descriptor row.

In [12]:
BASE_CODE = np.full(256, 255, dtype=np.uint8)
for base, code_value in ((ord("A"), 0), (ord("C"), 1), (ord("G"), 2), (ord("T"), 3)):
    BASE_CODE[base] = code_value

def make_descriptor_lookup(table, k, feature_names):
    lookup = np.empty((4**k, len(feature_names)), dtype=np.float32)
    seen = np.zeros(4**k, dtype=bool)
    for row in table[["SEQUENCE", *feature_names]].itertuples(index=False, name=None):
        sequence, *values = row
        code_value = 0
        for base in sequence.encode("ascii"):
            code_value = (code_value << 2) | int(BASE_CODE[base])
        lookup[code_value] = np.asarray(values, dtype=np.float32)
        seen[code_value] = True
    if not seen.all():
        raise ValueError(f"The {k}-mer lookup is missing sequence identifiers.")
    return lookup

tetramer_lookup = make_descriptor_lookup(tetramer_table, 4, FEATURES_CLEAN_TETRAMER)
hexamer_lookup = make_descriptor_lookup(hexamer_table, 6, FEATURES_CLEAN_HEXAMER)
print("Sequence-keyed lookup tables created:", tetramer_lookup.shape, hexamer_lookup.shape)


Sequence-keyed lookup tables created: (256, 36) (4096, 40)


## 4. Map windows and average descriptor values

For each 10-mer, the seven contiguous 4-mer windows and five contiguous 6-mer windows are encoded in their original left-to-right order. Descriptor values are looked up by sequence key and averaged separately within each window length. The implementation follows the two-bit encoding and averaging loops used by `make_predictors` in `scripts/eda_shape_affinity_adjusted.py`; composition columns used only for the EDA are not added to the machine-learning feature matrix.

In [14]:
def mean_window_descriptors(sequences, k, lookup):
    sequences = np.asarray(sequences, dtype=object)
    bases = np.frombuffer("".join(sequences).encode("ascii"), dtype=np.uint8).reshape(len(sequences), 10)
    n_windows = 10 - k + 1
    averages = np.zeros((len(sequences), lookup.shape[1]), dtype=np.float32)
    for start in range(n_windows):
        window_codes = np.zeros(len(sequences), dtype=np.uint16)
        for position in range(start, start + k):
            window_codes = (window_codes << 2) | BASE_CODE[bases[:, position]]
        averages += lookup[window_codes]
    averages /= n_windows
    return averages

tetramer_feature_names = [f"tetramer__{name}" for name in FEATURES_CLEAN_TETRAMER]
hexamer_feature_names = [f"hexamer__{name}" for name in FEATURES_CLEAN_HEXAMER]
model_feature_names = tetramer_feature_names + hexamer_feature_names
assert len(tetramer_feature_names) == 36
assert len(hexamer_feature_names) == 40
assert len(model_feature_names) == len(set(model_feature_names)) == 76
print("Feature columns per output:", len(model_feature_names))


Feature columns per output: 76


In [15]:
# Check the window order on one example against direct sequence-key lookup.
example_sequence = "AACGTACGTT"
example_tetramer = mean_window_descriptors([example_sequence], 4, tetramer_lookup)[0]
example_hexamer = mean_window_descriptors([example_sequence], 6, hexamer_lookup)[0]
tetra_by_sequence = tetramer_table.set_index("SEQUENCE")
hexa_by_sequence = hexamer_table.set_index("SEQUENCE")
expected_tetramer = np.mean([
    tetra_by_sequence.loc[example_sequence[start:start + 4], FEATURES_CLEAN_TETRAMER].to_numpy(dtype=np.float32)
    for start in range(7)
], axis=0, dtype=np.float32)
expected_hexamer = np.mean([
    hexa_by_sequence.loc[example_sequence[start:start + 6], FEATURES_CLEAN_HEXAMER].to_numpy(dtype=np.float32)
    for start in range(5)
], axis=0, dtype=np.float32)
np.testing.assert_allclose(example_tetramer, expected_tetramer, rtol=0, atol=1e-7)
np.testing.assert_allclose(example_hexamer, expected_hexamer, rtol=0, atol=1e-7)
print("Window-order check passed for 7 tetramer and 5 hexamer windows.")


Window-order check passed for 7 tetramer and 5 hexamer windows.


## 5. Preserve labels and export one file per TF

Each output row retains `TF`, selected `Cycle`, source `Kmer`, `Affinity`, `ObservedCount` and `SE`, followed by 76 mean descriptor features. Tetramer and hexamer feature names have explicit prefixes to avoid collisions (for example, `tetramer__SHIFT` and `hexamer__SHIFT`).

The output is a gzip-compressed CSV (`.csv.gz`), which pandas can read directly with `pd.read_csv`. Chunked processing preserves input row order and avoids loading all 11 datasets into memory at once. Outputs are written to `outputs/selex_ml_ready/`; no original data files are modified. The descriptors remain on their original scales. Any model-specific scaling should be fitted later using training data only.

In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_OUTPUT_COLUMNS = [
    "TF", "Cycle", "Kmer", "Affinity", "ObservedCount", "SE", *model_feature_names
]

def validate_target_chunk(chunk, tf):
    missing = set(TARGET_COLUMNS) - set(chunk.columns)
    if missing:
        raise KeyError(f"{tf}: source file is missing columns {sorted(missing)}")
    sequences = chunk["Kmer"].astype(str)
    valid_sequences = sequences.str.fullmatch(r"[ACGT]{10}", na=False)
    if not valid_sequences.all():
        raise ValueError(f"{tf}: invalid or non-10-mer sequence identifier found.")
    for column in ("ObservedCount", "Affinity", "SE"):
        values = pd.to_numeric(chunk[column], errors="raise").to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            raise ValueError(f"{tf}: non-finite or missing {column} values found.")
    return sequences.to_numpy()

def export_tf_dataset(tf, cycle, source_path):
    output_path = OUTPUT_DIR / f"{tf}_cycle{cycle}_10mer_descriptors.csv.gz"
    seen_sequences = set()
    row_count = 0
    first_chunk = True
    with gzip.open(output_path, mode="wt", newline="") as compressed_file:
        reader = pd.read_csv(source_path, sep="\t", usecols=TARGET_COLUMNS, chunksize=CHUNK_SIZE)
        for labels in reader:
            sequences = validate_target_chunk(labels, tf)
            duplicate_ids = [sequence for sequence in sequences if sequence in seen_sequences]
            if duplicate_ids:
                raise ValueError(f"{tf}: duplicated Kmer identifier: {duplicate_ids[0]}")
            seen_sequences.update(sequences)

            tetra_values = mean_window_descriptors(sequences, 4, tetramer_lookup)
            hexa_values = mean_window_descriptors(sequences, 6, hexamer_lookup)
            descriptor_values = np.concatenate([tetra_values, hexa_values], axis=1)
            if not np.isfinite(descriptor_values).all():
                raise ValueError(f"{tf}: non-finite averaged descriptor value found.")

            output_chunk = pd.DataFrame({
                "TF": tf,
                "Cycle": cycle,
                "Kmer": sequences,
                "Affinity": labels["Affinity"].to_numpy(),
                "ObservedCount": labels["ObservedCount"].to_numpy(),
                "SE": labels["SE"].to_numpy(),
            })
            output_chunk = pd.concat([
                output_chunk,
                pd.DataFrame(descriptor_values, columns=model_feature_names),
            ], axis=1)
            if output_chunk.columns.tolist() != EXPECTED_OUTPUT_COLUMNS:
                raise AssertionError(f"{tf}: unexpected output columns or order.")
            if output_chunk.isna().any().any():
                raise ValueError(f"{tf}: missing value found before export.")
            output_chunk.to_csv(compressed_file, index=False, header=first_chunk)
            first_chunk = False
            row_count += len(output_chunk)

    if row_count != len(seen_sequences):
        raise AssertionError(f"{tf}: output row count and unique sequence count differ.")
    return output_path, row_count


In [18]:
export_records = []
for tf, cycle in TF_CYCLES.items():
    output_path, n_rows = export_tf_dataset(tf, cycle, source_paths[tf])
    export_records.append({
        "TF": tf,
        "Cycle": cycle,
        "Rows": n_rows,
        "Columns": len(EXPECTED_OUTPUT_COLUMNS),
        "Descriptor features": len(model_feature_names),
        "Unique Kmers": n_rows,
        "Output file": str(output_path.relative_to(ROOT)),
    })
    print(f"{tf}: exported {n_rows:,} rows to {output_path.relative_to(ROOT)}")
    gc.collect()

export_summary = pd.DataFrame(export_records)
display(export_summary)


Gata4: exported 126,624 rows to outputs/selex_ml_ready/Gata4_cycle4_10mer_descriptors.csv.gz


Zfp24: exported 406,165 rows to outputs/selex_ml_ready/Zfp24_cycle5_10mer_descriptors.csv.gz


arid3: exported 273,249 rows to outputs/selex_ml_ready/arid3_cycle6_10mer_descriptors.csv.gz


hoxd1: exported 887,634 rows to outputs/selex_ml_ready/hoxd1_cycle3_10mer_descriptors.csv.gz


nkx3: exported 963,575 rows to outputs/selex_ml_ready/nkx3_cycle6_10mer_descriptors.csv.gz


pax3-like: exported 427,916 rows to outputs/selex_ml_ready/pax3-like_cycle5_10mer_descriptors.csv.gz


... (2 outputs left out)

telf: exported 691,492 rows to outputs/selex_ml_ready/telf_cycle5_10mer_descriptors.csv.gz


uncxb: exported 234,129 rows to outputs/selex_ml_ready/uncxb_cycle5_10mer_descriptors.csv.gz


xbpc: exported 463,571 rows to outputs/selex_ml_ready/xbpc_cycle5_10mer_descriptors.csv.gz


,TF,Cycle,Rows,Columns,Descriptor features,Unique Kmers,Output file
0,Gata4,4,126624,82,76,126624,outputs/selex_ml_ready/Gata4_cycle4_10mer_desc...
1,Zfp24,5,406165,82,76,406165,outputs/selex_ml_ready/Zfp24_cycle5_10mer_desc...
2,arid3,6,273249,82,76,273249,outputs/selex_ml_ready/arid3_cycle6_10mer_desc...
3,hoxd1,3,887634,82,76,887634,outputs/selex_ml_ready/hoxd1_cycle3_10mer_desc...
4,nkx3,6,963575,82,76,963575,outputs/selex_ml_ready/nkx3_cycle6_10mer_descr...
5,pax3-like,5,427916,82,76,427916,outputs/selex_ml_ready/pax3-like_cycle5_10mer_...
6,six1,5,944350,82,76,944350,outputs/selex_ml_ready/six1_cycle5_10mer_descr...
7,tbx15,5,25991,82,76,25991,outputs/selex_ml_ready/tbx15_cycle5_10mer_desc...
8,telf,5,691492,82,76,691492,outputs/selex_ml_ready/telf_cycle5_10mer_descr...
9,uncxb,5,234129,82,76,234129,outputs/selex_ml_ready/uncxb_cycle5_10mer_desc...


## 6. Validate the exported datasets

After writing, each compressed file is read back in chunks. We verify the output schema, total dimensions, numeric descriptor types, missing/non-finite values and consistency of `TF`, `Cycle` and `Kmer`. The source-side duplicate check guarantees one record per sequence within each TF dataset.

In [20]:
def validate_export(path, expected_tf, expected_cycle, expected_rows):
    total_rows = 0
    missing_values = 0
    invalid_features = 0
    output_columns = None
    for chunk in pd.read_csv(path, compression="gzip", chunksize=CHUNK_SIZE):
        if output_columns is None:
            output_columns = chunk.columns.tolist()
        if chunk.columns.tolist() != EXPECTED_OUTPUT_COLUMNS:
            raise ValueError(f"Unexpected columns in {path.name}")
        if not chunk["TF"].eq(expected_tf).all() or not chunk["Cycle"].eq(expected_cycle).all():
            raise ValueError(f"TF/cycle metadata mismatch in {path.name}")
        if not chunk["Kmer"].astype(str).str.fullmatch(r"[ACGT]{10}").all():
            raise ValueError(f"Invalid Kmer found in {path.name}")
        missing_values += int(chunk.isna().sum().sum())
        feature_values = chunk[model_feature_names].to_numpy(dtype=np.float64)
        invalid_features += int((~np.isfinite(feature_values)).sum())
        total_rows += len(chunk)
    if total_rows != expected_rows or missing_values or invalid_features:
        raise ValueError(
            f"Export validation failed for {path.name}: rows={total_rows}, "
            f"missing={missing_values}, non-finite descriptors={invalid_features}"
        )
    return {
        "TF": expected_tf,
        "Rows": total_rows,
        "Columns": len(output_columns),
        "Descriptor features": len(model_feature_names),
        "Missing values": missing_values,
        "Non-finite descriptors": invalid_features,
        "Schema valid": output_columns == EXPECTED_OUTPUT_COLUMNS,
    }

export_validation = pd.DataFrame([
    validate_export(
        OUTPUT_DIR / f"{tf}_cycle{cycle}_10mer_descriptors.csv.gz",
        tf,
        cycle,
        int(export_summary.loc[export_summary.TF == tf, "Rows"].iloc[0]),
    )
    for tf, cycle in TF_CYCLES.items()
])
assert export_validation["Schema valid"].all()
assert export_validation["Missing values"].eq(0).all()
assert export_validation["Non-finite descriptors"].eq(0).all()
display(export_validation)


,TF,Rows,Columns,Descriptor features,Missing values,Non-finite descriptors,Schema valid
0,Gata4,126624,82,76,0,0,True
1,Zfp24,406165,82,76,0,0,True
2,arid3,273249,82,76,0,0,True
3,hoxd1,887634,82,76,0,0,True
4,nkx3,963575,82,76,0,0,True
5,pax3-like,427916,82,76,0,0,True
6,six1,944350,82,76,0,0,True
7,tbx15,25991,82,76,0,0,True
8,telf,691492,82,76,0,0,True
9,uncxb,234129,82,76,0,0,True


In [21]:
summary_path = OUTPUT_DIR / "export_validation_summary.csv"
export_validation.to_csv(summary_path, index=False)
print(f"Validation summary saved to {summary_path.relative_to(ROOT)}")
print(f"Total exported 10-mer rows: {int(export_validation.Rows.sum()):,}")
print(f"Feature matrix: {len(model_feature_names)} descriptors per 10-mer (36 tetramer + 40 hexamer).")


Validation summary saved to outputs/selex_ml_ready/export_validation_summary.csv
Total exported 10-mer rows: 5,444,696
Feature matrix: 76 descriptors per 10-mer (36 tetramer + 40 hexamer).


## 7. Result and downstream use

The files in `outputs/selex_ml_ready/` are one dataset per TF and selected cycle. Load a file with `pd.read_csv(path_to_file, compression="gzip")`. Keep `TF`, `Cycle` and `Kmer` as identifiers/grouping metadata; use `Affinity` as the response; retain `ObservedCount` and `SE` as provenance/quality fields. Do not include these metadata or label columns among the descriptor predictors. Apply any scaling or train/test splitting in the later modelling workflow, not during this handoff.

The exported files are derived outputs and can be regenerated by rerunning this notebook. The original SELEX and descriptor tables remain unchanged.